In [ ]:
import scanpy as sc
import rpy2.robjects as ro
from scipy.stats import median_abs_deviation as mad
from pathlib import Path
from src import *

%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

sc.set_figure_params(dpi=150, facecolor='white')
sc.settings.verbosity = 3

project_root = Path.cwd().parent
data_dir = project_root / "data"
results_dir = project_root / "results"
ro.r(f'renv::load("{project_root}")')

In [ ]:
adata = sc.read_h5ad(data_dir / "processed" / "00_combined_processed.h5ad")
epithelial = adata[adata.obs["lineage"] == "epithelial"].copy()

In [ ]:
epithelial = deviant_hvg(epithelial)

In [ ]:
plot_deviance(epithelial)

In [ ]:
harmony_embed(epithelial)
sc.pl.pca_overview(epithelial, color="sample_id")

In [ ]:
cluster_subset(epithelial, n_pcs=25, resolution=0.4, min_dist=0.8)

In [ ]:
sc.pl.umap(epithelial, color=["scDblFinder_score", "sample_id", "pct_counts_mt", "leiden"], ncols=2)

In [ ]:
epithelial_genes = {
    "at1": ["Aqp5", "Akap5", "Pdpn", "Muc1"],
    "at2": ["Sftpc", "Sftpb", "Nkx2-1"],
    "transitional": ["Cldn18", "Krt8", "Cldn4"],
    "club": ["Scgb1a1", "Cyp2f2"],
    "ciliated": ["Foxj1"],
    "proliferating": ["Mki67", "Top2a"]
}

sc.tl.dendrogram(epithelial, groupby="leiden", use_rep="X_pca_harmony")
sc.pl.matrixplot(
    epithelial,
    var_names=epithelial_genes,
    groupby="leiden",
    dendrogram=True,
    cmap="Reds",
    standard_scale="var"
)

In [ ]:
epithelial_celltypes = {
    "at1": ["0"],
    "at2": ["3"],
    "prolif at2": ["4", "5"],
    "ciliated": ["2"],
    "transitional": ["1"],
    "club": ["6"]
}

apply_celltypes(epithelial, epithelial_celltypes)

In [ ]:
sc.pl.umap(epithelial, color="celltype")

In [ ]:
epithelial.write_h5ad(data_dir / "processed" / "03_epithelial.h5ad")